# Загрузка корпуса pf2.ru

Докачивает правила PF2e с pf2.ru: sitemap → HTML (с кэшем на диске) → парсинг → чанки
в `{TAPA_DATA_DIR}/pf2e-data/chunks/*.jsonl`. Использует тот же код, что и
`app.cli` / `app.corpus` — ноутбук просто даёт живые прогресс-бары и позволяет
следить и останавливать/продолжать вручную, не читая лог.

**Перед запуском:** pf2.ru время от времени блокирует краулер (отвечает 403) и
отпускает сам через какое-то время. Ноутбук ждёт и пробует снова — почти как
`scripts/download_corpus.ps1`, только с барами вместо текстового лога.

**Резюмируемо:** уже скачанные страницы лежат в `html_cache` по хешу URL, поэтому
повторный запуск (в том числе после `KeyboardInterrupt` — можно просто остановить
выполнение ячейки) не перекачивает их заново.

⚠️ **Не запускай одновременно с `scripts/download_corpus.ps1`.** Оба процесса
пишут в один и тот же `chunks/<раздел>.jsonl` в режиме перезаписи — параллельный
запуск на одном разделе потеряет часть уже скачанного. Останови PowerShell-скрипт
(Ctrl+C в его окне) перед тем, как выполнять ячейки ниже.

**Ядро:** этот ноутбук рассчитан на венв `packages/pf2e-data/.venv` — в Jupyter
выбери ядро **TAPA: pf2e-data** (зарегистрировано через `ipykernel install`).

In [ ]:
import sys
from pathlib import Path


def find_package_root(start: Path, name: str = "pf2e-data") -> Path:
    """Works whether the notebook is opened from its own folder or the package root."""
    for candidate in [start, *start.parents]:
        if candidate.name == name and (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(f"Не нашёл корень пакета {name!r} рядом с {start}")


PACKAGE_ROOT = find_package_root(Path.cwd())
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

PACKAGE_ROOT

In [ ]:
import time
from datetime import datetime, timedelta

from tqdm.auto import tqdm

from app.core.config import settings
from app.corpus import SECTIONS
from app.pipeline import run as run_pipeline
from app.scraper import SiteBlockedError
from app.sitemap import fetch_sitemap_urls, filter_by_prefix

CHUNK_DIR = Path(settings.output_dir)
print(f"Данные: {CHUNK_DIR.parent}")
print(f"Разделы (в порядке запуска): {', '.join(SECTIONS)}")

## Текущее состояние

Сколько чанков уже лежит в каждом разделе против того, сколько страниц вообще
есть в sitemap — чтобы понимать, что докачивать. Один запрос к sitemap.xml,
без обхода страниц.

In [ ]:
def corpus_status() -> None:
    urls = fetch_sitemap_urls(settings.sitemap_url, user_agent=settings.user_agent)
    print(f"{'раздел':<14} {'чанков':>8}  {'страниц в sitemap':>18}")
    for section in SECTIONS:
        jsonl = CHUNK_DIR / f"{section}.jsonl"
        chunks = sum(1 for _ in jsonl.open(encoding="utf-8")) if jsonl.is_file() else 0
        pages = len(filter_by_prefix(urls, path_prefix=f"/{section}/"))
        print(f"{section:<14} {chunks:>8}  {pages:>18}")


corpus_status()

## Настройки прогона

`SECTIONS_TO_RUN` — какие разделы качать (по умолчанию все, в том же порядке,
что и `app.corpus.SECTIONS` — сперва маленькие, чтобы сбой был виден за минуты,
а не за часы). Можно сузить список, например `["equipment", "feats"]`, чтобы
докачать конкретный раздел.

`LIMIT` — ограничение страниц на раздел; `None` значит «весь раздел». Полезно
поставить небольшое число (20–30) для быстрой проверки, что всё работает,
прежде чем оставлять ноутбук на ночь.

`MAX_ATTEMPTS` / `PAUSE_MINUTES` — сколько раз пробовать раздел и сколько
ждать между попытками при блокировке. Пауза намеренно большая: частые
повторы — это как раз то, из-за чего блокировка и наступает.

In [ ]:
SECTIONS_TO_RUN = SECTIONS
LIMIT = None
MAX_ATTEMPTS = 20
PAUSE_MINUTES = 25

## Загрузка

Для каждого раздела — прогресс-бар по страницам. Если pf2.ru начинает отвечать
403 подряд (см. `app.scraper.SiteBlockedError` — считает подряд идущие отказы
и останавливает раздел, а не помечает все оставшиеся страницы как
недоступные), ноутбук показывает бар обратного отсчёта паузы и пробует раздел
снова — уже скачанные страницы возьмутся из кэша мгновенно.

Остановить можно в любой момент (стоп-кнопкой ячейки или `KeyboardInterrupt`) —
прогресс на диске не потеряется, повторный запуск этой ячейки продолжит с
недокачанных страниц.

In [ ]:
def ingest_with_retries(section: str, *, sitemap_urls: list[str]) -> Path | None:
    total = len(filter_by_prefix(sitemap_urls, path_prefix=f"/{section}/", limit=LIMIT))

    for attempt in range(1, MAX_ATTEMPTS + 1):
        label = section if attempt == 1 else f"{section} (попытка {attempt}/{MAX_ATTEMPTS})"
        with tqdm(total=total, desc=label, unit="стр") as bar:
            try:
                out_path = run_pipeline(
                    path_prefix=f"/{section}/", limit=LIMIT, on_page=lambda: bar.update(1)
                )
            except SiteBlockedError as exc:
                bar.set_postfix_str("заблокировано")
                if attempt == MAX_ATTEMPTS:
                    print(f"{section}: не одолели за {MAX_ATTEMPTS} попыток — {exc}")
                    return None
                wake = datetime.now() + timedelta(minutes=PAUSE_MINUTES)
                print(f"{section}: {exc}\n   пауза до {wake:%H:%M:%S}")
                for _ in tqdm(
                    range(PAUSE_MINUTES * 60), desc="пауза перед повтором", unit="с", leave=False
                ):
                    time.sleep(1)
                continue

        lines = sum(1 for _ in out_path.open(encoding="utf-8"))
        print(f"{section}: {lines} чанков -> {out_path}")
        return out_path

    return None


sitemap_urls = fetch_sitemap_urls(settings.sitemap_url, user_agent=settings.user_agent)

results: dict[str, Path | None] = {}
try:
    for section in tqdm(SECTIONS_TO_RUN, desc="всего разделов", unit="раздел"):
        results[section] = ingest_with_retries(section, sitemap_urls=sitemap_urls)
except KeyboardInterrupt:
    print(
        "Остановлено вручную. Скачанное осталось в html_cache и в *.jsonl — "
        "повторный запуск этой ячейки продолжит с недокачанного."
    )

## Итог

In [ ]:
corpus_status()